# Mini-TP 7 · Seguridad con SAIF — Streaming (Kafka)

**Operaciones de Aprendizaje Automático II · CEIA – FIUBA**

Este es **uno de los seis notebooks guiados** del Mini-TP 7. Cada uno asegura **una capa**
de las que construimos en el bimestre. **Elige este si tu modelo se expone por Streaming (Kafka)**
(o simplemente para practicar esta estrategia). Con completar **uno** alcanza para la entrega.

### SAIF en esta capa
**Riesgo:** **envenenamiento** por eventos inyectados y productores no autorizados.  
**Controles:** validación de esquema por evento · cuarentena (*dead-letter queue*) · autenticación de productor (firma HMAC).  
**Elementos SAIF:** automatizar defensas · detectar y actuar.

### Qué vas a hacer
1. Ver la **versión vulnerable** y por qué es un riesgo.
2. Aplicar el **control principal** (guiado, ya resuelto como ejemplo).
3. Completar los **`# TODO`** sobre tu propio modelo.
4. Escribir una **reflexión** (amenaza mitigada ↔ elemento de SAIF).

> Corre de punta a punta con **uv**, sin servicios externos.


### Dependencias
```bash
uv add pydantic
```

## 1. El stream y una versión VULNERABLE
Simulamos un stream de eventos (como en la Sesión 4). El consumidor vulnerable **confía en
todo lo que llega**: un evento envenenado (fuera de rango o con etiqueta inválida) entra sin
control y contamina lo que después se reentrena.

In [ ]:
import hmac, hashlib, json, random
random.seed(0)
N_FEATURES = 4

def evento(i, envenenado=False, firmado=True):
    feats = [random.gauss(0, 1) for _ in range(N_FEATURES)]
    label = random.randint(0, 1)
    if envenenado:
        feats[0] = 1e9          # valor absurdo (fuera de rango)
        label = 99              # etiqueta inválida
    ev = {"id": i, "features": feats, "label": label}
    return ev

stream = [evento(i) for i in range(8)] + [evento(99, envenenado=True)]
# consumidor vulnerable: acepta todo
aceptados_v = [e for e in stream]
print("Consumidor vulnerable aceptó", len(aceptados_v), "eventos, incluido el envenenado:",
      any(e["label"] == 99 for e in aceptados_v))

## 2. Control principal (GUIADO) — validación de esquema + cuarentena (DLQ) + firma HMAC

In [ ]:
from pydantic import BaseModel, Field, conlist, ValidationError

SECRETO_PRODUCTOR = b"clave-compartida-del-productor"   # en prod: desde el entorno

def firmar(ev: dict) -> str:
    cuerpo = json.dumps(ev, sort_keys=True).encode()
    return hmac.new(SECRETO_PRODUCTOR, cuerpo, hashlib.sha256).hexdigest()

class EventoValido(BaseModel):
    id: int
    features: conlist(float, min_length=N_FEATURES, max_length=N_FEATURES)
    label: int = Field(ge=0, le=1)        # rango de etiquetas válido

def consumir(ev, firma):
    # 1) autenticación del productor (firma HMAC)
    if not hmac.compare_digest(firma, firmar(ev)):
        return "dlq_firma"
    # 2) validación de esquema (frena envenenamiento por rango/tipo)
    try:
        EventoValido(**ev)
    except ValidationError:
        return "dlq_esquema"
    return "ok"

aceptados, dlq = [], []
for ev in [evento(i) for i in range(8)] + [evento(99, envenenado=True)]:
    firma = firmar(ev)
    # el evento envenenado además llega SIN firma válida (productor no autorizado)
    if ev["id"] == 99:
        firma = "firma-falsa"
    destino = consumir(ev, firma)
    (aceptados if destino == "ok" else dlq).append((ev["id"], destino))

print("Aceptados:", [i for i, _ in aceptados])
print("En cuarentena (DLQ):", dlq, "  <- el envenenado quedó aislado")

## 3. Tu entrega — completa los `# TODO`

In [ ]:
# TODO 1: adapta 'EventoValido' al esquema REAL de tus eventos
#   (tus features y rangos; usa Field(ge=, le=) donde tenga sentido).
# TODO 2: registra la PROCEDENCIA de cada evento aceptado
#   (productor, timestamp, hash) en un log de auditoría.
# TODO 3: agrega una política para la DLQ: alertar si entran demasiados
#   eventos en cuarentena en una ventana (posible ataque en curso).
print("Completa los TODO sobre tu propio stream.")

## 4. Reflexión
1. ¿Por qué la validación de esquema frena un ataque de envenenamiento?
2. ¿Qué agrega la firma HMAC que la validación por sí sola no da?
3. ¿Qué elemento de SAIF cubre tu control principal?

*(Escribe tu respuesta aquí.)*